In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/StudentsPerformance.csv")
print(df.shape)
print(df.head())
print(df.info)
print(df.describe())


(1000, 8)
   gender race/ethnicity parental level of education         lunch  \
0  female        group B           bachelor's degree      standard   
1  female        group C                some college      standard   
2  female        group B             master's degree      standard   
3    male        group A          associate's degree  free/reduced   
4    male        group C                some college      standard   

  test preparation course  math score  reading score  writing score  
0                    none          72             72             74  
1               completed          69             90             88  
2                    none          90             95             93  
3                    none          47             57             44  
4                    none          76             78             75  
<bound method DataFrame.info of      gender race/ethnicity parental level of education         lunch  \
0    female        group B           bachelor

In [3]:
#basic statistics
for col in ['math score','reading score','writing score']:
    print(f"\n === {col.upper()} ===")
    print(f"Mean: {df[col].mean():.2f}")
    print(f"Median: {df[col].median():.2f}")
    print(f"Mode: {df[col].mode()[0]:.2f}")
    print(f"Std: {df[col].std():.2f}")
    print(f"Min: {df[col].min()}")
    print(f"Max: {df[col].max()}")



 === MATH SCORE ===
Mean: 66.09
Median: 66.00
Mode: 65.00
Std: 15.16
Min: 0
Max: 100

 === READING SCORE ===
Mean: 69.17
Median: 70.00
Mode: 72.00
Std: 14.60
Min: 17
Max: 100

 === WRITING SCORE ===
Mean: 68.05
Median: 69.00
Mode: 74.00
Std: 15.20
Min: 10
Max: 100


In [4]:
# ============================================================
# TASK 2: DESCRIPTIVE STATISTICS
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_csv("data/StudentsPerformance.csv")
scores = ['math score', 'reading score', 'writing score']

# ----- Mean, Median, Mode, Std, Min, Max -----
stats = {}
for col in scores:
    stats[col] = {
        'Mean':   round(df[col].mean(), 2),
        'Median': df[col].median(),
        'Mode':   df[col].mode()[0],
        'Std':    round(df[col].std(), 2),
        'Min':    df[col].min(),
        'Max':    df[col].max()
    }

stats_df = pd.DataFrame(stats).T
print("=== DESCRIPTIVE STATISTICS ===")
print(stats_df)

=== DESCRIPTIVE STATISTICS ===
                Mean  Median  Mode    Std   Min    Max
math score     66.09    66.0  65.0  15.16   0.0  100.0
reading score  69.17    70.0  72.0  14.60  17.0  100.0
writing score  68.05    69.0  74.0  15.20  10.0  100.0


In [5]:
# ----- Mean vs Median Comparison -----
print("\n=== MEAN vs MEDIAN COMPARISON ===")
for col in scores:
    mean = df[col].mean()
    median = df[col].median()
    diff = mean - median
    skew = "left-skewed (outliers pulling down)" if diff < 0 else "right-skewed (outliers pulling up)" if diff > 0 else "symmetric"
    print(f"{col}: Mean={mean:.2f}, Median={median:.2f}, Diff={diff:.2f} → {skew}")


=== MEAN vs MEDIAN COMPARISON ===
math score: Mean=66.09, Median=66.00, Diff=0.09 → right-skewed (outliers pulling up)
reading score: Mean=69.17, Median=70.00, Diff=-0.83 → left-skewed (outliers pulling down)
writing score: Mean=68.05, Median=69.00, Diff=-0.95 → left-skewed (outliers pulling down)


In [6]:
# ----- GroupBy: Average scores by Gender -----
print("\n=== AVERAGE SCORES BY GENDER ===")
print(df.groupby('gender')[scores].mean().round(2))


=== AVERAGE SCORES BY GENDER ===
        math score  reading score  writing score
gender                                          
female       63.63          72.61          72.47
male         68.73          65.47          63.31


In [7]:
# ----- GroupBy: Average scores by Parental Education -----
print("\n=== AVERAGE SCORES BY PARENTAL LEVEL OF EDUCATION ===")
edu_order = ["some high school","high school","some college",
             "associate's degree","bachelor's degree","master's degree"]
result = df.groupby('parental level of education')[scores].mean().round(2)
result = result.reindex(edu_order)
print(result)


=== AVERAGE SCORES BY PARENTAL LEVEL OF EDUCATION ===
                             math score  reading score  writing score
parental level of education                                          
some high school                  63.50          66.94          64.89
high school                       62.14          64.70          62.45
some college                      67.13          69.46          68.84
associate's degree                67.88          70.93          69.90
bachelor's degree                 69.39          73.00          73.38
master's degree                   69.75          75.37          75.68


Female students score higher in reading and writing; males score higher in math.
Students whose parents have a master's degree score highest on average.
Mean ≈ Median for all scores, indicating roughly symmetric distributions.
Writing and reading scores are very closely correlated.
Students with "some high school" parental education have the lowest average scores.

In [8]:
#generating messy_students.csv
messy = df.copy()

np.random.seed(42)
n = len(messy)

# Inject missing values
for col in ['math score', 'reading score', 'writing score']:
    idx = np.random.choice(n, size=30, replace=False)
    messy.loc[idx, col] = np.nan

# Inject outliers
messy.loc[0, 'math score'] = 200
messy.loc[1, 'reading score'] = -10

# Inject duplicates
duplicates = messy.sample(15)
messy = pd.concat([messy, duplicates], ignore_index=True)

# Corrupt some strings
messy.loc[5, 'gender'] = 'Male'     # wrong capitalization
messy.loc[6, 'gender'] = ' female ' # extra spaces

messy.to_csv("data/messy_students.csv", index=False)
print("messy_students.csv created:", messy.shape)

messy_students.csv created: (1015, 8)


In [ ]:
messy = pd.read_csv("data/messy_students.csv")

print("=== SHAPE ===")
print(messy.shape)

print("\n=== INFO ===")
messy.info()

print("\n=== MISSING VALUES ===")
print(messy.isnull().sum())

print("\n=== DUPLICATES ===")
print(f"Duplicate rows: {messy.duplicated().sum()}")

print("\n=== UNIQUE VALUES IN GENDER ===")
print(messy['gender'].unique())

print("\n=== OUTLIERS (scores outside 0-100) ===")
for col in ['math score', 'reading score', 'writing score']:
    outliers = messy[(messy[col] < 0) | (messy[col] > 100)]
    print(f"{col}: {len(outliers)} outliers")